> **Archived (round 1, see plan_t6.md).** `item_gaps` and `n_steps` were removed from `GapRegressor` after this run, so this notebook no longer runs as written; the saved outputs are the record.

# Task 6 · improvements (plan_t6.md)

**Shipped:** `GapRegressor`, backtest A (each user's last train gap) **7,182,796**, backtest B
(one step further back) **7,191,137**; public **8.76M** (tier 3.5; 4.0 needs 8.5M, about −3%).

Options on `GapRegressor` (defaults = shipped model):
| option | plan | what |
|---|---|---|
| `item_gaps=True` | P2 | other readers' log gap *into* the next item (median, quartiles, count) |
| `coread=True` | P3 | readers of both the user's last item and the next item: count, share who read it after, quartiles of the delay |
| `n_steps=k` | P1 | k training targets per user (k-th last gap each), stacked |
| `n_bag=b` | P4 | b seeds (early-stopping split), log predictions averaged |

**Ship bar:** better on **both** backtests, by ≥ 2% (≈ −145k) for the combined variant.
Slow cells: each fit is on the full data; `n_steps` multiplies the feature cost.

In [1]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd

from assignment_1.modules.models import GapModel, GapRegressor, split_last
from assignment_1.modules.util import load_csv

In [2]:
DATA_DIR = "../data"  # notebook runs from notebooks/

train = load_csv("task_6_train.csv", DATA_DIR)
test = load_csv("task_6_test.csv", DATA_DIR)

# backtest A: hold out each user's last gap; B: one step further back
trA, valA = split_last(train)
trB, valB = split_last(trA)
print(f"A: {len(valA):,} users | B: {len(valB):,} users")

A: 55,403 users | B: 55,403 users


## Helper
`results` accumulates every run, so cells can be re-run or skipped independently.

In [3]:
results = {}

def run(name: str, backtests: str = "A", **kw) -> pd.DataFrame:
    """Fit GapRegressor(**kw) on each backtest's history; MAE on its held-out gaps."""
    row = results.setdefault(name, {})
    for b in backtests:
        tr, val = (trA, valA) if b == "A" else (trB, valB)
        t0 = time.time()
        pred = GapRegressor(base=GapModel(k=10, lam=1.5), **kw).fit(tr).predict(val)
        row[f"mae_{b}"] = float(np.mean(np.abs(pred - val["target"].to_numpy())))
        row[f"sec_{b}"] = round(time.time() - t0)
    return table()

def table() -> pd.DataFrame:
    T = pd.DataFrame(results).T
    for b in "AB":
        if f"mae_{b}" in T and "shipped" in results and f"mae_{b}" in results["shipped"]:
            T[f"Δ%_{b}"] = 100 * (T[f"mae_{b}"] / results["shipped"][f"mae_{b}"] - 1)
    return T.round(2)

## E1. Shipped model (expect A 7,182,796 · B 7,191,137)

In [4]:
run("shipped", "AB")

,mae_A,sec_A,mae_B,sec_B,Δ%_A,Δ%_B
shipped,7182795.63,19.0,7191136.89,19.0,0.0,0.0


## E2. New item features (P2, P3)

In [5]:
run("+ item_gaps", item_gaps=True)

,mae_A,sec_A,mae_B,sec_B,Δ%_A,Δ%_B
shipped,7182795.63,19.0,7191136.89,19.0,0.00,0.0
+ item_gaps,7185263.21,20.0,NaN,NaN,0.03,NaN


In [6]:
run("+ coread", coread=True)

,mae_A,sec_A,mae_B,sec_B,Δ%_A,Δ%_B
shipped,7182795.63,19.0,7191136.89,19.0,0.00,0.0
+ item_gaps,7185263.21,20.0,NaN,NaN,0.03,NaN
+ coread,7115770.40,42.0,NaN,NaN,-0.93,NaN


In [7]:
run("+ item_gaps + coread", item_gaps=True, coread=True)

,mae_A,sec_A,mae_B,sec_B,Δ%_A,Δ%_B
shipped,7182795.63,19.0,7191136.89,19.0,0.00,0.0
+ item_gaps,7185263.21,20.0,NaN,NaN,0.03,NaN
+ coread,7115770.40,42.0,NaN,NaN,-0.93,NaN
+ item_gaps + coread,7115651.47,44.0,NaN,NaN,-0.93,NaN


## E3. More training targets (P1)
Uses the feature set from E2 that did best; edit `FEATS` if a single one won.

In [8]:
FEATS = dict(item_gaps=True, coread=True)   # <- set from E2
run("feats, n_steps=2", n_steps=2, **FEATS)

,mae_A,sec_A,mae_B,sec_B,Δ%_A,Δ%_B
shipped,7182795.63,19.0,7191136.89,19.0,0.00,0.0
+ item_gaps,7185263.21,20.0,NaN,NaN,0.03,NaN
+ coread,7115770.40,42.0,NaN,NaN,-0.93,NaN
+ item_gaps + coread,7115651.47,44.0,NaN,NaN,-0.93,NaN
"feats, n_steps=2",7127153.86,65.0,NaN,NaN,-0.77,NaN


In [9]:
run("feats, n_steps=3", n_steps=3, **FEATS)

,mae_A,sec_A,mae_B,sec_B,Δ%_A,Δ%_B
shipped,7182795.63,19.0,7191136.89,19.0,0.00,0.0
+ item_gaps,7185263.21,20.0,NaN,NaN,0.03,NaN
+ coread,7115770.40,42.0,NaN,NaN,-0.93,NaN
+ item_gaps + coread,7115651.47,44.0,NaN,NaN,-0.93,NaN
"feats, n_steps=2",7127153.86,65.0,NaN,NaN,-0.77,NaN
"feats, n_steps=3",7145327.67,88.0,NaN,NaN,-0.52,NaN


## E4. Seed bagging (P4)

In [10]:
STEPS = 2   # <- best from E3 (1 if neither helped)
run(f"feats, n_steps={STEPS}, n_bag=5", n_steps=STEPS, n_bag=5, **FEATS)

,mae_A,sec_A,mae_B,sec_B,Δ%_A,Δ%_B
shipped,7182795.63,19.0,7191136.89,19.0,0.00,0.0
+ item_gaps,7185263.21,20.0,NaN,NaN,0.03,NaN
+ coread,7115770.40,42.0,NaN,NaN,-0.93,NaN
+ item_gaps + coread,7115651.47,44.0,NaN,NaN,-0.93,NaN
"feats, n_steps=2",7127153.86,65.0,NaN,NaN,-0.77,NaN
"feats, n_steps=3",7145327.67,88.0,NaN,NaN,-0.52,NaN
"feats, n_steps=2, n_bag=5",7111673.13,78.0,NaN,NaN,-0.99,NaN


## E5. Confirm the best variant on backtest B
Ship only if it also wins on B (an independent target per user).

In [11]:
BEST = dict(n_steps=STEPS, n_bag=5, **FEATS)   # <- the best row above
run("best", "AB", **BEST)

,mae_A,sec_A,mae_B,sec_B,Δ%_A,Δ%_B
shipped,7182795.63,19.0,7191136.89,19.0,0.00,0.00
+ item_gaps,7185263.21,20.0,NaN,NaN,0.03,NaN
+ coread,7115770.40,42.0,NaN,NaN,-0.93,NaN
+ item_gaps + coread,7115651.47,44.0,NaN,NaN,-0.93,NaN
"feats, n_steps=2",7127153.86,65.0,NaN,NaN,-0.77,NaN
"feats, n_steps=3",7145327.67,88.0,NaN,NaN,-0.52,NaN
"feats, n_steps=2, n_bag=5",7111673.13,78.0,NaN,NaN,-0.99,NaN
best,7111673.13,78.0,7095230.52,74.0,-0.99,-1.33


## E6. New feature coverage and importance (sanity check)
Share of validation rows where each new feature is defined, and the drop in backtest-A error
when that feature is shuffled (larger = more useful).

In [12]:
from sklearn.inspection import permutation_importance

m = GapRegressor(base=GapModel(k=10, lam=1.5), item_gaps=True, coread=True).fit(trA)
F = m.features(trA, valA)
new = [c for c in F if c.startswith(("ig_", "co_"))]
print(F[new].notna().mean().round(3).to_string())

yA = np.log1p(valA["target"].to_numpy())
imp = permutation_importance(m.reg_, F, yA, scoring="neg_mean_absolute_error",
                             n_repeats=3, random_state=0)
pd.Series(imp.importances_mean, index=F.columns).sort_values(ascending=False).head(20).round(4)

ig_med           0.933
ig_q25           0.933
ig_q75           0.933
ig_n             0.933
co_n             0.933
co_frac_after    0.468
co_q25           0.392
co_med           0.392
co_q75           0.392


it_frac_after    0.0992
it_q5            0.0601
t_last_yr        0.0470
bound_days       0.0445
to_end_d         0.0437
it_q10           0.0397
co_q25           0.0305
ig_n             0.0119
lag_0            0.0118
it_first         0.0081
it_q25           0.0074
co_n             0.0047
lg_q25           0.0029
it_after         0.0029
it_q50           0.0027
it_n             0.0023
co_frac_after    0.0022
lg_med5          0.0022
ig_med           0.0022
lg_med10         0.0020
dtype: float64

## Decision
- If `best` wins on both A and B (≥ 2% on A): set `TASK6_REGRESSOR_PARAMS = BEST` in
  `tasks/task6.py`, then `make check`, `make changes`, and upload `dist/task_6_predictions.csv`.
- Validation → public has run at about 1.22× (7.18M → 8.76M), so −3% on public needs roughly
  −3% on backtest A.
- Record the numbers in plan_t6.md.